# Chapter 5: Sequence Models — Vanilla RNN & LSTM from Scratch

Welcome to Chapter 5! In this chapter, we transition from static spatial data (images and tabular points) to **dynamic temporal sequences** (natural text, time-series, audio, and language).

### The Big Question: The Goldfish Memory Problem 🐟
Imagine reading a murder mystery novel, but after reading each single word, your memory is completely wiped clean.
- If you read the word *"bank"*, does it mean a financial institution or the side of a river?
- Without the previous words (*"He deposited cash into the..."* vs *"The fisherman sat on the river..."*), you have zero context.

Feedforward networks (MLPs) and CNNs treat every input independently — they have no concept of time or sequence. **Recurrent Neural Networks (RNNs)** introduce a recurrent loop: passing a hidden state vector $h_t$ from time step to time step like a diary entry. **Long Short-Term Memory (LSTM)** networks take this even further with a protected additive highway that prevents memories from fading over long time horizons.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Add repository root to path
REPO_ROOT = Path("..").resolve().parent
sys.path.insert(0, str(REPO_ROOT))

from src.rnn import CharVocab, VanillaRNN, LSTM
print("Successfully imported CharVocab, VanillaRNN, and LSTM from scratch!")


## 1. Character Tokenization & Vocabulary Mapping

In character-level language modeling, we treat every character (letters, spaces, punctuation, digits) as an atomic token. Let's inspect our Shakespeare corpus.


In [ ]:
corpus_path = REPO_ROOT / "data" / "shakespeare.txt"
with open(corpus_path, "r", encoding="utf-8") as f:
    text_corpus = f.read()

vocab = CharVocab(text_corpus)
print(f"Corpus size: {len(text_corpus):,} characters")
print(f"Vocab size: {vocab.vocab_size} distinct characters")
print("Sample vocabulary characters:", vocab.chars[:20])

sample_phrase = "First Citizen: Speak!"
encoded = vocab.encode(sample_phrase)
decoded = vocab.decode(encoded)
print(f"\nPhrase:  {sample_phrase}")
print(f"Encoded: {encoded}")
print(f"Decoded: {decoded}")
assert sample_phrase == decoded


## 2. Vanilla RNN Architecture & The Recurrent Loop

At every time step $t$, the Vanilla RNN receives an input vector $x_t$ (1-hot encoded) and the previous hidden state $h_{t-1}$:

$$h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h)$$
$$\hat{y}_t = W_{hy} h_t + b_y$$
$$p_t = \text{softmax}(\hat{y}_t)$$

Let's run a forward pass on a 4-character sequence and inspect the shapes.


In [ ]:
hidden_dim = 32
rnn = VanillaRNN(vocab_size=vocab.vocab_size, hidden_dim=hidden_dim, seed=42)

input_chars = vocab.encode("Hell")
target_chars = vocab.encode("ello")
h0 = np.zeros((hidden_dim, 1))

loss, h_final, cache = rnn.forward(input_chars, target_chars, h0)
print(f"Vanilla RNN Sequence Cross-Entropy Loss: {loss:.4f}")
print(f"Final hidden state shape: {h_final.shape}")

# Analytical Backpropagation Through Time (BPTT)
grads = rnn.backward(target_chars, cache)
print("Gradient shapes computed via BPTT:")
for k, v in grads.items():
    print(f"  {k:5s}: shape {str(v.shape):15s} norm {np.linalg.norm(v):.4f}")


## 3. Why Vanilla RNNs Suffer: Vanishing & Exploding Gradients

Notice that when propagating gradients back $T$ time steps in a Vanilla RNN:
$$\frac{\partial h_T}{\partial h_1} = \prod_{t=2}^T \frac{\partial h_t}{\partial h_{t-1}} = \prod_{t=2}^T \text{diag}(1 - h_t^2) W_{hh}^T$$

Multiplying by $W_{hh}$ repeatedly over $T$ steps behaves like taking $W_{hh}^T$. If the eigenvalues of $W_{hh}$ are strictly less than 1, the gradient shrinks exponentially to zero (Vanishing Gradients). If greater than 1, it blows up exponentially (Exploding Gradients).

### The LSTM Solution: The Protected Conveyor Belt 🛈
The **Long Short-Term Memory (LSTM)** replaces the recurrent matrix multiplication with an additive highway called the **Cell State** ($C_t$):

$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

Because the update is **addition** rather than matrix multiplication, gradients can flow back dozens or hundreds of time steps with zero decay!


In [ ]:
lstm = LSTM(vocab_size=vocab.vocab_size, hidden_dim=hidden_dim, seed=42)
c0 = np.zeros((hidden_dim, 1))

loss_lstm, h_lstm, c_lstm, cache_lstm = lstm.forward(input_chars, target_chars, h0, c0)
print(f"LSTM Initial Cross-Entropy Loss: {loss_lstm:.4f}")
print(f"LSTM Hidden State shape: {h_lstm.shape}, Cell State shape: {c_lstm.shape}")

grads_lstm = lstm.backward(target_chars, cache_lstm)
print("LSTM Analytical Gradients:")
for k, v in grads_lstm.items():
    print(f"  {k:8s}: shape {str(v.shape):15s} norm {np.linalg.norm(v):.4f}")


## 4. Side-by-Side Training Comparison (Vanilla RNN vs LSTM)

Let's train both models side-by-side on consecutive chunks of Shakespeare text using the Adagrad optimizer.


In [ ]:
encoded_full = vocab.encode(text_corpus)
n_steps = 300
seq_len = 25
lr = 0.1
H = 64

rnn_model = VanillaRNN(vocab.vocab_size, hidden_dim=H, seed=42)
lstm_model = LSTM(vocab.vocab_size, hidden_dim=H, seed=42)

m_rnn = {k: np.zeros_like(v) for k, v in rnn_model.params.items()}
m_lstm = {k: np.zeros_like(v) for k, v in lstm_model.params.items()}

h_r = np.zeros((H, 1))
h_l = np.zeros((H, 1))
c_l = np.zeros((H, 1))

history_rnn = []
history_lstm = []
smooth_r = -np.log(1.0 / vocab.vocab_size) * seq_len
smooth_l = -np.log(1.0 / vocab.vocab_size) * seq_len

ptr = 0
for step in range(n_steps):
    if ptr + seq_len + 1 >= len(encoded_full):
        ptr = 0
        h_r, h_l, c_l = np.zeros((H, 1)), np.zeros((H, 1)), np.zeros((H, 1))

    in_seq = encoded_full[ptr : ptr + seq_len]
    tgt_seq = encoded_full[ptr + 1 : ptr + seq_len + 1]

    # RNN Step
    loss_r, h_r, cache_r = rnn_model.forward(in_seq, tgt_seq, h_r)
    grads_r = rnn_model.backward(tgt_seq, cache_r)
    for k in rnn_model.params:
        m_rnn[k] += grads_r[k] ** 2
        rnn_model.params[k] -= lr * grads_r[k] / (np.sqrt(m_rnn[k]) + 1e-8)

    # LSTM Step
    loss_l, h_l, c_l, cache_l = lstm_model.forward(in_seq, tgt_seq, h_l, c_l)
    grads_l = lstm_model.backward(tgt_seq, cache_l)
    for k in lstm_model.params:
        m_lstm[k] += grads_l[k] ** 2
        lstm_model.params[k] -= lr * grads_l[k] / (np.sqrt(m_lstm[k]) + 1e-8)

    smooth_r = smooth_r * 0.99 + loss_r * 0.01
    smooth_l = smooth_l * 0.99 + loss_l * 0.01
    history_rnn.append(smooth_r / seq_len)
    history_lstm.append(smooth_l / seq_len)

    ptr += seq_len

print(f"Final 300-step RNN Loss:  {history_rnn[-1]:.3f} nats/char")
print(f"Final 300-step LSTM Loss: {history_lstm[-1]:.3f} nats/char")


In [ ]:
plt.figure(figsize=(9, 4.5), dpi=150)
plt.plot(history_rnn, label="Vanilla RNN", color="#f87171", lw=2)
plt.plot(history_lstm, label="LSTM", color="#38bdf8", lw=2)
plt.xlabel("Training Step")
plt.ylabel("Loss per Character")
plt.title("Training Loss Comparison: Vanilla RNN vs LSTM")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()


## 5. Live Character Generation & Temperature Control

When generating text, we pass the predicted logits $y$ through softmax with a temperature parameter $T$:
$$p_i = \frac{e^{y_i / T}}{\sum_j e^{y_j / T}}$$

- **Low Temperature ($T = 0.3$):** Sharper probabilities, safer, predictable, repeats common words.
- **Balanced ($T = 0.7$):** Natural diversity with coherent word structures.
- **High Temperature ($T = 1.3$):** Flattened probabilities, highly creative, but higher probability of gibberish.


In [ ]:
seed_char = vocab.encode("F")[0]
for temp in [0.3, 0.7, 1.2]:
    sampled_indices = lstm_model.sample(h_l, c_l, seed_char, n_chars=160, temperature=temp, seed=42)
    generated_text = vocab.decode(sampled_indices)
    print(f"=== Temperature {temp:.1f} ===")
    print(generated_text)
    print("-" * 50)
